In [9]:
!pip install plotly pandas


In [10]:
!pip install --upgrade nbformat ipywidgets plotly


# 🏛️ Congressional Bills - Exploratory Data Analysis (EDA)

This notebook explores the dataset of U.S. Congressional Bills (118th Congress). 
We will analyze the reality of legislative success by diving into five specific questions. We will use the `bills_clean.csv` dataset, which contains feature-engineered signals.


In [11]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json

# Set default plotly theme
import plotly.io as pio
pio.templates.default = "plotly_white"

# Load the cleaned dataset
df = pd.read_csv('../bills_clean.csv')

# Derived columns for easier plotting
df['is_naming_bill_str'] = df['is_naming_bill'].map({1: 'Naming Bill', 0: 'Substantive Legislation'})

# Update bypassed committee labeling based on context (often procedural/administrative)
df['bypassed_committee_str'] = df['bypassed_committee'].map({1: 'Procedural/Administrative (Bypassed)', 0: 'Standard Committee Review'})

df['passed_str'] = df['passed'].map({1: 'Passed', 0: 'Failed'})
df['text_length_log'] = df['clean_text'].fillna("").apply(len)

print(f"Total Bills Loaded: {len(df)}")
print(f"Overall Pass Rate: {df['passed'].mean():.1%}")


Total Bills Loaded: 2224
Overall Pass Rate: 15.5%


## 1. Are "naming bills" inflating Congress's productivity stats?

Post office renaming bills ("To designate the facility of the United States Postal Service located at...") are non-controversial and bipartisan by nature. They pass at a staggering rate. If we strip them out, the story of Congressional productivity completely changes.


In [12]:
naming_stats = df.groupby(['sponsor_chamber', 'sponsor_party', 'is_naming_bill_str'])['passed'].agg(['mean', 'count']).reset_index()
# Filter for major parties and reasonable sample sizes
naming_stats = naming_stats[naming_stats['sponsor_party'].isin(['D', 'R'])]

fig = px.bar(
    naming_stats,
    x='sponsor_chamber',
    y='mean',
    color='sponsor_party',
    facet_col='is_naming_bill_str',
    barmode='group',
    title="Pass Rate by Chamber & Party: Naming Bills vs. Substantive Legislation",
    labels={"mean": "Passage Rate", "sponsor_chamber": "Chamber", "sponsor_party": "Party"},
    color_discrete_map={"D": "#3b82f6", "R": "#ef4444"}
)
fig.update_layout(yaxis_tickformat='.0%')
fig.show()


## 2. Does bill length predict success?

Short passed bills are often real, brief, and focused legislation (e.g. the "Full Faith and Credit Act"). Long bills are massive substantive packages that likely die in committee from complexity.


In [13]:
# Group by text length ranges (bins)
bins = [0, 2000, 5000, 15000, 50000, float('inf')]
labels = ['Very Short (<2k chars)', 'Short (2k-5k)', 'Medium (5k-15k)', 'Long (15k-50k)', 'Massive (>50k)']
df['length_category'] = pd.cut(df['text_length_log'], bins=bins, labels=labels)

length_stats = df.groupby('length_category')['passed'].agg(['mean', 'count']).reset_index()

fig = px.bar(
    length_stats, 
    x="length_category", 
    y="mean", 
    text="count",
    title="Passage Rate by Bill Length",
    labels={"mean": "Passage Rate", "length_category": "Bill Length (Characters)", "count": "Total Bills Introduced"},
    color="mean",
    color_continuous_scale="Teal"
)
fig.update_traces(texttemplate='%{text} bills', textposition='outside')
fig.update_layout(yaxis_tickformat='.0%')
fig.show()


## 3. The "Committee Count" Paradox

It appears that bills that bypass committees completely have a high pass rate. However, these are often procedural resolutions (e.g. "Electing officers of the House" or "fixing the hour of daily meeting"). They bypass committees because they are internal administrative housekeeping.


In [14]:
fasttrack_stats = df.groupby('bypassed_committee_str')['passed'].agg(['mean', 'count']).reset_index()

fig = px.bar(
    fasttrack_stats,
    x='bypassed_committee_str',
    y='mean',
    text='count',
    title="Pass Rate: Standard Committee Review vs. Administrative Fast-Track",
    labels={"mean": "Passage Rate", "bypassed_committee_str": "Committee Process"},
    color="bypassed_committee_str",
    color_discrete_sequence=["#8b5cf6", "#f59e0b"]
)
fig.update_traces(texttemplate='%{text} bills', textposition='outside')
fig.update_layout(yaxis_tickformat='.0%', showlegend=False)
fig.show()


## 4. Which committees are gatekeepers vs. fast lanes?

The committee you land in is almost destiny. Some committees (like Homeland Security & Oversight, which handle post office naming) have massive pass rates, while others (Energy & Commerce) act as strict gatekeepers.


In [17]:
import ast

# Extract committee names and track passes
committee_data = []
for _, row in df.iterrows():
    if pd.isna(row['committee_names_str']):
        continue
    try:
        
        committees = [c.strip() for c in row['committee_names_str'].split('|') if c.strip()]

        for c in committees:
            committee_data.append({'committee': c, 'passed': row['passed']})
    except:
        pass

cdf = pd.DataFrame(committee_data)
c_stats = cdf.groupby('committee')['passed'].agg(['mean', 'count']).reset_index()

# Filter for committees with at least 15 bills to remove noise
c_stats = c_stats[c_stats['count'] >= 15].sort_values('mean', ascending=False)

fig = px.bar(
    
    pd.concat([c_stats.head(10), c_stats.tail(10)]), # Top 10 and Bottom 10

    y='committee',
    x='mean',
    color='mean',
    orientation='h',
    title="Top 10 Fast Lanes vs. Top 10 Gatekeeper Committees (Min. 15 bills)",
    labels={"mean": "Passage Rate", "committee": "Committee"},
    color_continuous_scale="RdYlGn"
)
fig.update_layout(yaxis_tickformat='.0%', height=600)
fig.show()


## 5. Do Senate Democrats punch above their weight?

Despite introducing fewer bills than Senate Republicans, Democratic senators often have a higher substantive pass rate — suggesting they might be more selective about what they introduce.


In [18]:
# Look only at substantive legislation (no naming bills) in the Senate
senate_substantive = df[(df['sponsor_chamber'] == 'Senate') & (df['is_naming_bill'] == 0)]
senate_party_stats = senate_substantive.groupby('sponsor_party')['passed'].agg(['mean', 'count']).reset_index()
senate_party_stats = senate_party_stats[senate_party_stats['sponsor_party'].isin(['D', 'R'])]

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "domain"}, {"type": "bar"}]], subplot_titles=["Volume of Substantive Bills Introduced", "Passage Rate of Substantive Bills"])

# Pie chart for volume
fig.add_trace(go.Pie(labels=senate_party_stats['sponsor_party'], values=senate_party_stats['count'], 
                     marker_colors=["#3b82f6", "#ef4444"], name="Volume"), row=1, col=1)

# Bar chart for passage rate
fig.add_trace(go.Bar(x=senate_party_stats['sponsor_party'], y=senate_party_stats['mean'], 
                     marker_color=["#3b82f6", "#ef4444"], name="Pass Rate"), row=1, col=2)

fig.update_layout(title_text="Senate: Democrats vs. Republicans (Substantive Legislation Only)", showlegend=False)
fig.update_yaxes(tickformat='.0%', row=1, col=2)
fig.show()
